# Avaliação Prática 1 part 2

In [24]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from sklearn.model_selection import validation_curve
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

SEMENTE = 42
pd.set_option("display.width", 100)

## Exercício 1 (4,0 pontos)
Uma empresa de hortifrúti mantém estufas climatizadas e quer prever o consumo diário de energia de cada estufa (aquecimento no frio e refrigeração no calor) a partir da previsão do tempo para o dia seguinte, para contratar a demanda de energia com antecedência. Ela reuniu 150 dias de operação no arquivo estufas_energia.csv, com as colunas:

In [25]:
df_soja = pd.read_csv("estufas_energia.csv")
display(df_soja.head())

,id_estufa,temperatura_externa_c,umidade_relativa_pct,custo_energia_rs_dia,consumo_kwh_dia
0,1,33.3,67,210.97,256.6
1,2,12.3,38,155.50,190.9
2,3,23.7,57,136.87,168.7
3,4,30.0,73,168.77,203.9
4,5,27.8,57,150.68,183.6


### a) Identifique o tipo de problema (supervisionado ou não, e a natureza da variável a prever) e escolha o algoritmo que utilizará, justificando a escolha com base no que se espera da relação entre temperatura e consumo. [0,5]

R: Trata-se de um problema supervisionado, pois temos o histórico com os rótulos/respostas (consumo) já conhecidos para treinar o modelo. A natureza da variável a prever (`consumo_kwh_dia`) é numérica, logo, se trata de um problema de regressão. Algoritmo escolhido: `Polynomial Features` (Regressão Polinomial), pois se trata de um algoritmo eficiente e capaz de estimar um valor numérico baseando-se em múltiplas variáveis de entrada.

### b) Defina quais colunas serão usadas como entradas do modelo e qual será a saída. Justifique a utilização ou a exclusão de cada coluna da tabela acima, tendo em mente o momento em que a previsão será feita. [0,5]

R: Entradas: `temperatura_externa_c`, `umidade_relativa_pct`. Saída: `consumo_kwh_dia`. Exclusões: `id_estufa` é apenas um identificador único, não possui influência sobre o consumo; `custo_energia_rs_dia`: o custo só será conhecida após saber o consumo. Justificativas das inclusões: `temperatura_externa_c`, `umidade_relativa_pct`: são todas as informações disponíveis ou projetadas antes ou durante o consumo das estufas e que têm impacto direto no consumo.

In [26]:
X_estufa = df_soja[["temperatura_externa_c", "umidade_relativa_pct"]]
y_estufa = df_soja["consumo_kwh_dia"]

### c) Padronize as colunas temperatura_externa_c e umidade_relativa_pct com StandardScaler. Explique em que momento o StandardScaler deve ser ajustado (fit) em relação à divisão dos dados da validação e por quê, e indique em que ponto da sequência de etapas (padronização, geração dos termos polinomiais, regressão) ele deve ficar.

R: O `StandardScaler` deve ser ajustado apenas na massa de treino, e isso deve ser feito após a divisão da base de dados entre treino e validação. Isso ocorre para evitar o vazamento de dados. O uso de um `Pipeline` no Scikit-Learn resolve essa questão automaticamente durante a validação cruzada.

### d) Com validação cruzada de 5 partições (embaralhadas, random_state=42), teste os graus de 1 a 8. Apresente, para cada grau, o RMSE médio de treino e o RMSE médio de validação. Escolha o grau e justifique a escolha, indicando quais graus apresentam subajuste e quais apresentam sobreajuste. [1,0]

R: Foi escolhido o grau 4 por possuir a melhor média, sendo o "ponto alto" da função polinomial.


In [27]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X_estufa, y_estufa, test_size=0.30, random_state=SEMENTE
)
print(f"treino: {X_treino.shape[0]} dados   teste: {X_teste.shape[0]} dados")

preprocessor_estufa = ColumnTransformer([
        ('padroniza', StandardScaler(), ['temperatura_externa_c', 'umidade_relativa_pct'])
])

validacao = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)

pipeline_estufa = Pipeline([
    ('preprocessor', preprocessor_estufa),
    ('modelo', PolynomialFeatures(degree=2)),
])

graus = [1, 2, 3, 4, 6, 8]
linhas = []

for grau in graus:
    poly = PolynomialFeatures(degree=grau, include_bias=False)
    Xtr_p = poly.fit_transform(X_treino)
    Xte_p = poly.transform(X_teste)

    modelo = LinearRegression().fit(Xtr_p, y_treino)
    pred_tr = modelo.predict(Xtr_p)
    pred_te = modelo.predict(Xte_p)

    linhas.append({
        "grau": grau,
        "RMSE_treino": mean_squared_error(y_treino, pred_tr) ** 0.5,
        "RMSE_validacao": mean_squared_error(y_teste, pred_te) ** 0.5,
    })


tabela_graus = pd.DataFrame(linhas)
tabela_graus.round(4)


treino: 105 dados   teste: 45 dados


,grau,RMSE_treino,RMSE_validacao
0,1,56.3711,49.9955
1,2,12.6579,13.0298
2,3,12.5471,12.9045
3,4,22.9054,21.1121
4,6,32.7035,28.6900
5,8,37.5580,35.6787


## Questão 3 (discursiva — 2,0 pontos)
Sobre os dois algoritmos desta prova e sobre a escolha de hiperparâmetros: (1) explique como o valor de K no K-NN e o grau na regressão polinomial controlam a complexidade do modelo, relacionando valores extremos de cada um com subajuste e sobreajuste; (2) explique por que K e o grau não devem ser escolhidos pelo desempenho no conjunto de treino e como escolhê-los de forma adequada; (3) cite uma limitação do K-NN e uma limitação da regressão polinomial ao fazer previsões para dados novos. Dê, ainda, um exemplo de situação em que a regressão polinomial seria preferível à regressão linear simples. Responda em até 15 linhas

R: (1) - No K-NN, K baixo gera alta complexidade (sobreajuste) e K alto causa subajuste. Na regressão polinomial, grau baixo causa subajuste e grau alto causa sobreajuste por alta variabilidade. (2) O conjunto de treino não deve ser usado pois favorece a memorização de ruídos (complexidade extrema zera o erro de treino). A escolha adequada exige validação cruzada ou conjunto de validação. (3) O K-NN é lento em bases grandes e sofre com a maldição da dimensionalidade. A regressão polinomial extrapola mal, gerando curvas instáveis fora do intervalo de treino. Exemplo: A regressão polinomial é preferível à linear simples quando há relações não lineares, como a relação entre temperatura e taxa de reação química.